# 第 1 周第 2 天练习 —— 网页抓取 + Ollama 分块学习路线图

## 练习目标（理念）

把 **网页正文抓取** 与 **本地/云端 Ollama 聊天** 串起来：给定学习清单 URL，抓取内容后按块喂给模型，生成「主题列表 + 学习路线图 + 练习题」。

- **输入**：一个练习清单网站 URL（示例为 NeetCode 150）
- **输出**：Markdown 形式的学习路线与练习建议
- **关键技巧**：长文本先 `chunk_text` 分块，再逐块 `ollama.chat`，最后把各块摘要再汇总一次

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| HTTP 抓取 + BeautifulSoup | `fetch_website_contents` |
| `messages`（system / user） | `messages_for` |
| 本地/云端模型（Ollama） | `ollama.chat(model=MODEL_NAME, ...)` |
| 长上下文处理 | 按字符近似分块再汇总 |

## 怎么跑

1. 确保已安装 `ollama` Python 包，且本机/云端已有 `MODEL_NAME` 对应模型
2. 从上到下运行；主流程在「RUN」段的 `display_summary(...)`
3. 可把 URL 换成你自己的学习清单页再试


## 说明

本笔记本核心逻辑集中在下一个代码单元格：配置 → 抓取 → 分块 → 调 Ollama → Markdown 展示。


In [ ]:
# ========== 导入：抓取、解析、展示、日志 ==========

# 导入 ollama：Python 客户端，调用本机或云端 Ollama 的 chat 接口
import ollama
# 导入 requests：用 HTTP GET 拉取网页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的文档树，便于去噪取正文
from bs4 import BeautifulSoup
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown
from IPython.display import Markdown, display
# 导入 logging：把抓取/分块进度打到日志，便于排查
import logging

# 配置日志级别为 INFO：info/error 都会显示
logging.basicConfig(level=logging.INFO)

# ========== CONFIG：模型名与分块大小 ==========

# Ollama 模型名：需与本机/云端已安装的模型字符串一致
MODEL_NAME = "gemma4:31b-cloud"
# 分块大小（字符近似，当作 token 安全上限的粗估计）
MAX_CHUNK_SIZE = 3000  # tokens safety approx


# ========== 抓取网页正文 ==========

def fetch_website_contents(url: str) -> str:
    # try：网络/HTTP 可能失败，失败时记日志并返回空串
    try:
        # 自定义 User-Agent：部分站点会拒默认爬虫头
        headers = {
            "User-Agent": "Mozilla/5.0 (compatible; AI-Summarizer/1.0)"
        }

        # GET 网页；timeout=10 避免长时间挂起
        response = requests.get(url, headers=headers, timeout=10)
        # 非 2xx 时抛 HTTPError
        response.raise_for_status()

        # 用 html.parser 解析响应文本
        soup = BeautifulSoup(response.text, "html.parser")

        # 去掉脚本/样式/导航等噪声标签，减少无关文本进入模型
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.extract()

        # 抽出纯文本：空格分隔、strip 去首尾空白
        text = soup.get_text(separator=" ", strip=True)

        # 成功日志
        logging.info("Website content fetched successfully.")
        return text

    # 捕获 requests 相关异常（超时、连接失败、HTTP 错误等）
    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")
        return ""


# ========== 把长文本按固定长度切成多块 ==========

def chunk_text(text: str, chunk_size: int = MAX_CHUNK_SIZE):
    # 列表推导：每 chunk_size 个字符一块（简化版，不按句子边界）
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]


# ========== 为单块文本构造 messages ==========

def messages_for(text_chunk: str):
    return [
        {
            "role": "system",
            # system prompt 保留英文：发给模型的任务指令，改译会改变行为
            "content": "You are a helpful AI tutor. I want you to give everything every topics which are given in the website with the bullet points as well. Generate a roadmap and study plan for all those topics. And practice questions for every topics in the sheet."
        },
        {
            "role": "user",
            # user：当前这一块的网页正文
            "content": text_chunk
        }
    ]


# ========== 对单个 chunk 调 Ollama ==========

def summarize_chunk(text_chunk: str) -> str:
    try:
        # ollama.chat：同步聊天；返回 dict，正文在 message.content
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages_for(text_chunk)
        )
        return response["message"]["content"]

    except Exception as e:
        # 任意异常都记日志，并返回空串让上层继续
        logging.error(f"Summarization error: {e}")
        return ""


# ========== 主流程：抓取 → 分块 → 逐块摘要 → 再汇总 ==========

def summarize(url: str) -> str:
    # 先抓网页正文
    text = fetch_website_contents(url)

    # 抓取失败则直接返回提示字符串（含 emoji，保持原样）
    if not text:
        return "❌ Failed to fetch website content."

    # 按 MAX_CHUNK_SIZE 切开
    chunks = chunk_text(text)

    logging.info(f"Total chunks: {len(chunks)}")

    # 逐块调用模型，收集各块输出
    summaries = []
    for i, chunk in enumerate(chunks):
        logging.info(f"Processing chunk {i + 1}/{len(chunks)}...")
        summary = summarize_chunk(chunk)
        summaries.append(summary)

    # 把各块摘要拼成一大段，再喂给模型做「最终精炼」
    final_summary = summarize_chunk(" ".join(summaries))

    return final_summary


# ========== 展示：summarize 后用 Markdown 渲染 ==========

def display_summary(url: str):
    summary = summarize(url)
    display(Markdown(summary))


# ========== RUN：作为脚本入口时跑示例 URL ==========

# Jupyter 里直接跑本格也会执行到这里（__name__ 常为 "__main__"）
if __name__ == "__main__":
    display_summary("https://neetcode.io/practice/practice/neetcode150")


In [ ]:
# （空单元格）可在此继续试验：例如换 URL 再调 display_summary(...)
